# 05 — Partitioning and Caching in Apache Spark

## Why Partitions Matter

> **Core principle:** Partitions are the unit of parallelism in Spark. Every transformation and action
> operates on individual partitions, and Spark schedules one task per partition per executor core.

A DataFrame is physically stored as a collection of **partitions** distributed across the cluster.
Understanding how to control partition count and distribution is the single most impactful
performance tuning lever available to a Spark developer.

### What you will learn
1. How to inspect and change partition counts with `repartition` and `coalesce`.
2. How to partition data by a column value for data locality.
3. How Spark's storage levels work and when to cache vs persist.
4. How to measure the real speedup from caching with timing comparisons.

**Cluster:** `spark://spark-master:7077`  
**Data dir:** `/home/jovyan/data`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import time

# Build a SparkSession connected to our standalone cluster.
# spark.sql.shuffle.partitions controls how many partitions result from a shuffle operation.
# The default is 200, which is too high for small datasets — we set it to 8 here.
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("05-Partitioning-and-Caching")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

# Set log level to WARN so the notebook output stays readable
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

In [ ]:
# -----------------------------------------------------------------------
# Create a large DataFrame to make partitioning behaviour visible.
# spark.range(n) generates a single LongType column called 'id'.
# We add a 'category' column by mapping id modulo 4 to a letter.
# -----------------------------------------------------------------------

NUM_ROWS = 1_000_000

df = (
    spark.range(NUM_ROWS)           # id: 0 … 999_999
    .withColumn(
        "category",
        # Map id % 4  ->  'A', 'B', 'C', or 'D'
        F.element_at(
            F.array(
                F.lit("A"), F.lit("B"), F.lit("C"), F.lit("D")
            ),
            (F.col("id") % 4).cast("int") + 1   # element_at is 1-indexed
        )
    )
    .withColumn("value", (F.rand(seed=42) * 1000).cast("double"))
)

print(f"Schema         : {df.schema.simpleString()}")
print(f"Row count      : {df.count():,}")
print(f"Default parts  : {df.rdd.getNumPartitions()}")
df.show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------
# repartition(n)  — full shuffle, produces exactly n balanced partitions
# coalesce(n)     — no shuffle, merges existing partitions locally;
#                   fast but may produce skewed partition sizes
# -----------------------------------------------------------------------

# --- repartition: full shuffle -------------------------------------------
df_rep = df.repartition(8)   # triggers a full shuffle across the network
print(f"After repartition(8)  : {df_rep.rdd.getNumPartitions()} partitions")

# --- coalesce: local merge -----------------------------------------------
# Reducing from default to 2 without a shuffle — cheaper but less balanced
df_coal = df.coalesce(2)     # no shuffle; just merges tasks on the same node
print(f"After coalesce(2)     : {df_coal.rdd.getNumPartitions()} partitions")

# --- Inspect actual partition sizes via mapPartitionsWithIndex ------------
def partition_size(idx, iterator):
    """Returns (partition_index, row_count) for each partition."""
    count = sum(1 for _ in iterator)
    yield (idx, count)

print("\nrepartition(8) partition sizes:")
for part_id, size in sorted(
    df_rep.rdd.mapPartitionsWithIndex(partition_size).collect()
):
    print(f"  Partition {part_id}: {size:>9,} rows")

print("\ncoalesce(2) partition sizes:")
for part_id, size in sorted(
    df_coal.rdd.mapPartitionsWithIndex(partition_size).collect()
):
    print(f"  Partition {part_id}: {size:>9,} rows")

## repartition vs coalesce — When to Use Each

Both methods change the partition count, but they do so very differently:

| Feature | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Network shuffle | **Yes** — full shuffle | **No** — local merge only |
| Partition balance | Always balanced | Can be skewed |
| Allowed direction | Increase **or** decrease | Decrease **only** |
| Partition by column | Yes — `repartition(n, col)` | No |
| Typical use case | Before a wide transformation that needs balance | Saving a small result to fewer files |
| Cost | High (shuffle = disk + network I/O) | Low |

### Rule of thumb
- Use **`repartition`** when you need balanced data distribution before joins or aggregations, or when you want to increase partition count.
- Use **`coalesce`** when you are writing a final small result to storage and just want fewer output files — the imbalance is acceptable because there is no further computation.

In [ ]:
# -----------------------------------------------------------------------
# Custom partitioning by column value
#
# repartition("category") hashes the category column and places all rows
# with the same category value into the same partition.  This is called
# "data locality" — subsequent groupBy("category") operations no longer
# need a shuffle because co-located rows are already on the same node.
# -----------------------------------------------------------------------

# Partition by the category column — Spark decides how many partitions
# to create (one per distinct hash bucket, up to spark.sql.shuffle.partitions)
df_by_cat = df.repartition("category")

print(f"Partitions after repartition('category'): {df_by_cat.rdd.getNumPartitions()}")

# Verify: each partition should contain rows from only ONE category
def categories_in_partition(idx, iterator):
    rows = list(iterator)
    cats = set(r["category"] for r in rows)
    yield (idx, sorted(cats), len(rows))

print("\nCategories per partition (first 8):")
results = (
    df_by_cat.rdd
    .mapPartitionsWithIndex(categories_in_partition)
    .collect()
)
for part_id, cats, count in sorted(results)[:8]:
    print(f"  Partition {part_id}: categories={cats}, rows={count:,}")

# -----------------------------------------------------------------------
# You can also combine a partition count with a column:
#   df.repartition(4, "category")  -> exactly 4 partitions, hashed by category
# -----------------------------------------------------------------------
df_4_cat = df.repartition(4, "category")
print(f"\nrepartition(4, 'category')  -> {df_4_cat.rdd.getNumPartitions()} partitions")

## Storage Levels — Choosing the Right One

When you call `cache()` or `persist()`, Spark stores a materialized copy of the DataFrame
according to the chosen **storage level**. Picking the wrong level wastes memory or slows
deserialization.

| Storage Level | Memory | Disk | Serialized | Replication | Best When |
|---|:---:|:---:|:---:|:---:|---|
| `MEMORY_ONLY` | Yes | No | No | 1x | Data fits in RAM; fast CPU; JVM heap available |
| `MEMORY_AND_DISK` | Yes | Yes (overflow) | No | 1x | Data may exceed RAM; recomputation is expensive |
| `DISK_ONLY` | No | Yes | Yes | 1x | RAM is scarce; recomputation is very expensive |
| `MEMORY_ONLY_SER` | Yes | No | Yes | 1x | RAM is tight; willing to pay CPU for deserialization |
| `MEMORY_AND_DISK_SER` | Yes | Yes (overflow) | Yes | 1x | Balance: smaller footprint + disk fallback |
| `MEMORY_ONLY_2` | Yes | No | No | 2x | Fault-tolerant; data must survive a node failure |
| `OFF_HEAP` | Off-heap | No | Yes | 1x | Avoid GC pressure on large cached datasets |

### Quick Decision Guide
- **Default `cache()`** uses `MEMORY_AND_DISK` (since Spark 3.x for DataFrames).
- Use **`MEMORY_ONLY`** when your dataset is small and you re-use it many times.
- Use **`MEMORY_AND_DISK`** when you are unsure — it is the safe default.
- Use **`_SER` variants** when you are memory-constrained and willing to pay CPU overhead.
- Use **`DISK_ONLY`** as a last resort — sequential disk I/O is still faster than recomputing a long lineage.

In [ ]:
# -----------------------------------------------------------------------
# df.cache() is a shorthand for persist(StorageLevel.MEMORY_AND_DISK)
# for DataFrames in Spark 3.x.
#
# IMPORTANT: cache() and persist() are LAZY — they do not store anything
# until an ACTION (like count()) forces the DataFrame to be computed.
# -----------------------------------------------------------------------

from pyspark.sql.utils import AnalysisException

# Use the balanced repartitioned DataFrame for caching demos
df_work = df.repartition(8)

# --- Timing WITHOUT cache ------------------------------------------------
t0 = time.time()
count_1 = df_work.count()   # triggers full computation from scratch
t_no_cache = time.time() - t0
print(f"Without cache — count={count_1:,}  time={t_no_cache:.3f}s")

# --- Cache the DataFrame -------------------------------------------------
df_work.cache()   # registers the caching intent (still lazy)

# First action after cache() — computes AND stores the partitions
t0 = time.time()
count_2 = df_work.count()   # computes + caches all partitions
t_first_cached = time.time() - t0
print(f"First cached   — count={count_2:,}  time={t_first_cached:.3f}s  (write to cache)")

# Subsequent action — served entirely from cached partitions
t0 = time.time()
count_3 = df_work.count()   # reads from cache — no recomputation
t_cached = time.time() - t0
print(f"From cache     — count={count_3:,}  time={t_cached:.3f}s  (read from cache)")

# --- Verify cached status ------------------------------------------------
# spark.catalog.isCached() only works on a *named* relation. Querying a name
# that was never registered raises AnalysisException (TABLE_OR_VIEW_NOT_FOUND),
# so we guard the call and fall back to registering a temp view first.
try:
    is_cached = spark.catalog.isCached("__df_work")   # not registered — will raise
    print(f"isCached('__df_work') : {is_cached}")
except AnalysisException as e:
    print("isCached('__df_work') raised AnalysisException — the name is not "
          "registered in the catalog. Registering a temp view first...")

df_work.createOrReplaceTempView("df_work_view")
spark.catalog.cacheTable("df_work_view")           # cache by table name
print(f"\nisCached('df_work_view') : {spark.catalog.isCached('df_work_view')}")

# Speedup summary
speedup = t_no_cache / t_cached if t_cached > 0 else float('inf')
print(f"\nSpeedup from cache : {speedup:.1f}x  ({t_no_cache:.3f}s -> {t_cached:.3f}s)")

In [ ]:
# -----------------------------------------------------------------------
# persist() with explicit StorageLevel
#
# Use this when you need fine-grained control over where and how
# Spark stores the cached data.
# -----------------------------------------------------------------------

# First unpersist the earlier cache so we start clean
df_work.unpersist()
print(f"After unpersist — isCached('df_work_view'): {spark.catalog.isCached('df_work_view')}")

# Create a fresh DataFrame for this demo
df_persist = df.repartition(8)

# --- MEMORY_AND_DISK — safe default, spills to disk if heap is full ------
df_persist.persist(StorageLevel.MEMORY_AND_DISK)
_ = df_persist.count()   # trigger materialization
print(f"\npersist(MEMORY_AND_DISK) triggered")
print(f"  Partitions cached: {df_persist.rdd.getNumPartitions()}")

# --- DISK_ONLY — useful when RAM is very tight ---------------------------
df_disk = df.repartition(4)
df_disk.persist(StorageLevel.DISK_ONLY)
_ = df_disk.count()
print(f"\npersist(DISK_ONLY) triggered for df_disk")

# --- MEMORY_ONLY_SER — smaller memory footprint via serialization --------
df_ser = df.repartition(4)
df_ser.persist(StorageLevel.MEMORY_ONLY_SER)
_ = df_ser.count()
print(f"persist(MEMORY_ONLY_SER) triggered for df_ser")

# --- Explicit unpersist — always free cached resources when done ---------
df_persist.unpersist(blocking=True)  # blocking=True waits for deallocation
df_disk.unpersist(blocking=True)
df_ser.unpersist(blocking=True)
spark.catalog.uncacheTable("df_work_view")

print("\nAll caches released.")
print("Tip: Spark also auto-evicts LRU partitions when memory pressure is high.")

In [ ]:
# -----------------------------------------------------------------------
# Clean up — always stop the SparkSession at the end of a notebook
# to release executor resources back to the cluster.
# -----------------------------------------------------------------------
spark.stop()
print("SparkSession stopped. Resources released.")